In [1]:
import logging
import sys
import os

# Add the project root directory to Python path using our project initialization module
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..')))
# Configure logging to show INFO level messages
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
os.listdir()

['.ipynb_checkpoints', 'app', 'data', 'example', 'main.py', 'logs']

#### 加载数据集

In [2]:
import pandas as pd
df_raw = pd.read_excel('./data/0828.xlsx')

In [3]:
# 提取所需的四列数据
selected_columns = ['事件编号', '登记时间', '事件标题', '事件描述','诉求小类']
df_selected = df_raw[selected_columns]

# 重命名列名为英文标识符
df_selected.rename(columns={
    '事件编号': 'id',
    '登记时间': 'date',
    '事件标题': 'title',
    '事件描述': 'text',
    '诉求小类': 'req_type'
}, inplace=True)

# 创建最终的DataFrame

final_document = df_selected[df_selected['req_type'].isin(['保险监管', '失业保险','工商保险','生育保险', '城乡居民医疗保险', '职工医疗保险','城乡居民养老保险'])]
# 显示新DataFrame的前几行
final_document.shape

/tmp/ipykernel_336/3583253555.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected.rename(columns={


(2249, 5)

In [4]:
# 抽取知识图谱的操作
# from graphrag.prompts.index.extract_graph import GRAPH_EXTRACTION_PROMPT
# from openai import OpenAI
# client = OpenAI(
#     api_key="sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
#     base_url="http://172.16.2.158/api/llm2/v1"
# )

import asyncio
from openai import AsyncOpenAI
from graphrag.prompts.index.extract_graph import GRAPH_EXTRACTION_PROMPT  # Assuming this is available; replace with your prompt if needed
from app.core.graph_extractor_async import GraphExtractor  # As per your import path; adjust if the class is defined elsewhere
client = AsyncOpenAI(
    api_key="sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
    base_url="http://172.16.2.158/api/llm2/v1"
)

In [5]:
# 定义进度回调函数
def progress_callback(current, total):
    print(f"\r处理进度: {current}/{total} ({current/total*100:.1f}%)", end='', flush=True)

extractor = GraphExtractor(
    client=client,
    model_name="Qwen3-32B",  # Or your preferred model
    prompt=f"{GRAPH_EXTRACTION_PROMPT} 保持原始语料的语境.如果给你中文材料，你要保留中文的命名 //no_thinking",  # Use the imported prompt
    max_gleanings=1,  # Allow up to 2 additional gleanings for more details
    on_progress=progress_callback  # 添加进度回调
)

In [6]:
df = final_document['title'] + final_document['text']
texts = df.to_list()
texts[:5]

['咨询投诉支付宝的问题支付宝的蚂蚁保险给其扣除了费用，但其未购买该保险，咨询如何投诉支付宝。',
 '求助如何获取游泳救生员补贴我在今年5月经过厦门游泳协会考取了游泳救生员证件，本人也工作3年，结果弄了很久的材料发现原来的失业保险中技能提升补贴又不行了，打电话联系了失业保险的单位也是被一下推去岛内考取证件的区，现在又推回来同安。',
 '咨询投诉蚂蚁保险的问题咨询投诉蚂蚁保险的问题，通话过程中，诉求人主动挂机。',
 '咨询实名关联参保的问题其小孩在厦门市同安区预参保，咨询如何办理实名关联参保。',
 '咨询医保个人账户清退的问题其之前在厦门参保，医保个人账户余额有500多元钱，咨询是否可以提取现金。']

In [ ]:

# Prompt variables (customize delimiters or entity types if needed)
prompt_variables = {
    "entity_types": ["实体名称", "地点", "事件"],  # Matches above
}

# Call the extractor asynchronously
result = await extractor(texts, prompt_variables)

# Print the extracted graph details
graph = result.output
print("\nExtracted Nodes:")
for node, data in graph.nodes(data=True):
    print(f"- {node}: {data}")

print("\nExtracted Edges:")
for source, target, data in graph.edges(data=True):
    print(f"- {source} -> {target}: {data}")



2025-09-19 00:52:03,307 - httpx - INFO - HTTP Request: POST http://172.16.2.158/api/llm2/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-19 00:52:09,044 - httpx - INFO - HTTP Request: POST http://172.16.2.158/api/llm2/v1/chat/completions "HTTP/1.1 200 OK"
